# 05_identifiability — Identifiability and recovery of the fitted parameters

**Manuscript:** Methods 'Identifiability and recovery'; Supplementary S12 (tab:identifiability); S13 control leave-one-out magnitude anchor.

Four pre-specified checks on the selected fits with PCA-basis voxel synthesis: Test 1 parameter recovery at the selected optimum (`param_recovery_voxel.py`), Test 2a noise floor at a (0, 0) ground truth and Test 2b control pseudo-CVD refits (`null_within_hc_loo.py`), Test 2c colour-label permutation (`null_label_permutation_block.py`); `analyze_verification.py` assembles the verdict matrix with Benjamini-Hochberg correction. These scripts import the model and loss code of `04_distortion_model/scripts`.

**How to read this notebook.** Every code cell loads committed result files from `results/` and compares the values it derives with the numbers printed in the manuscript (`V.check`). A check passes when the produced value equals the printed one at the printed precision, or satisfies the stated relation. Quantities that have no committed artifact are recorded as pointers (`V.flag`) rather than silently omitted. The last cell tallies the checks and writes `_checks_05_identifiability.json`, which `run_notebooks.py` collects into `REPORT.md`.

Provenance: built by `tools/public_repo/build.py` of the development repository (commit 53c81c2); manuscript source in `../paper/`; check list in `../MANIFEST.md`; code map in `../MAP.md`.

**Source and code map**

| Result file | Producing script | What it holds |
|---|---|---|
| `results/param_recovery_voxel_v6_pca_v2.json` | `scripts/param_recovery_voxel.py` | Test 1: 7 donors x 20 noise draws at the selected optimum |
| `results/null_within_hc_loo_v6_pca.json` | `scripts/null_within_hc_loo.py` | Test 2a (origin null, B2) and Test 2b (control pseudo-CVD, B1) |
| `results/null_label_permutation_v6_pca.json` | `scripts/null_label_permutation_block.py` | Test 2c: 1,000 colour-label permutations |
| `results/verdict_matrix_v6_pca_v2.json` | `scripts/analyze_verification.py` | criteria, verdicts and BH correction |

In [1]:
import sys, json, csv
from pathlib import Path
sys.path.insert(0, str((Path.cwd() / ".." / "common").resolve()))
import numpy as np
from scipy import stats
import verify as V
from stats_helpers import crawford_howell, hedges_g, bh_fdr, wilson_interval
R = Path("results")
def J(name):
    with open(R / name) as f:
        return json.load(f)
HC = [f"sub-{i:02d}" for i in range(1, 8)]
CVD = {"deutan": "sub-08", "protan": "sub-09"}
ROIS = ["V1", "V2", "V3", "hV4"]
HUES = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "magenta"]

pr = J("param_recovery_voxel_v6_pca_v2.json")["cells"]
nl = J("null_within_hc_loo_v6_pca.json")["cells"]
vm = J("verdict_matrix_v6_pca_v2.json")["per_candidate"]
lp = J("null_label_permutation_v6_pca.json")
CAND = {"deutan": "S08-robust", "protan": "S09-primary"}

V.start("05_identifiability")

### Test 1: parameter recovery at the selected optimum (Supplementary S12, tab:identifiability)
Recovery fraction within 10 degrees on both axes and bias, aggregated over the seven control donors (20 noise draws each).

| id | manuscript | quantity | reported |
|---|---|---|---|
| 05.01 | S12 ¶3 | 7 donors x 20 draws = 140 samples per candidate | `(7, 20)` |
| 05.02 | tab:identifiability | deutan f_10 | `0.26` |
| 05.03 | tab:identifiability | protan f_10 | `0.14` |
| 05.04 | tab:identifiability | both fail the f_10 >= 0.5 criterion | `True` |
| 05.05 | tab:identifiability | deutan bias on beta_s (+16, median over donors) | `16.0` |
| 05.06 | tab:identifiability | deutan bias on beta_c (-4.7, mean over donors; the donor median is -4.0) | `-4.7` |
| 05.07 | tab:identifiability | protan bias on beta_s (+11) | `11.0` |
| 05.08 | tab:identifiability | protan bias on beta_c (-27) | `-27.0` |
| 05.09 | S12 ¶4 | deutan dominant-axis bias 4.7 against 16 on the non-dominant axis | `True` |

In [2]:
t1 = {}
for k, c in CAND.items():
    cells = [v for key, v in pr.items() if key.startswith(c + "_mag1.0")]
    t1[k] = dict(n=len(cells), f10=np.mean([v["frac_within_10deg"] for v in cells]), n_draw=cells[0]["n"],
                 bias_bs_median=np.median([v["bias_bs"] for v in cells]), bias_bc_median=np.median([v["bias_bc"] for v in cells]),
                 bias_bs_mean=np.mean([v["bias_bs"] for v in cells]), bias_bc_mean=np.mean([v["bias_bc"] for v in cells]))
    print(k, {a: round(b, 3) for a, b in t1[k].items()})
V.check('05.01', 'S12 ¶3 | 7 donors x 20 draws = 140 samples per candidate', (t1["deutan"]["n"], t1["deutan"]["n_draw"]), (7, 20), mode='eq')
V.check('05.02', 'tab:identifiability | deutan f_10', t1["deutan"]["f10"], 0.26, nd=2)
V.check('05.03', 'tab:identifiability | protan f_10', t1["protan"]["f10"], 0.14, nd=2)
V.check('05.04', 'tab:identifiability | both fail the f_10 >= 0.5 criterion', max(t1["deutan"]["f10"], t1["protan"]["f10"]) < 0.5, True, mode='eq')
V.check('05.05', 'tab:identifiability | deutan bias on beta_s (+16, median over donors)', t1["deutan"]["bias_bs_median"], 16.0, nd=0)
V.check('05.06', 'tab:identifiability | deutan bias on beta_c (-4.7, mean over donors; the donor median is -4.0)', t1["deutan"]["bias_bc_mean"], -4.7, nd=1)
V.check('05.07', 'tab:identifiability | protan bias on beta_s (+11)', t1["protan"]["bias_bs_median"], 11.0, nd=0)
V.check('05.08', 'tab:identifiability | protan bias on beta_c (-27)', t1["protan"]["bias_bc_median"], -27.0, nd=0)
V.check('05.09', 'S12 ¶4 | deutan dominant-axis bias 4.7 against 16 on the non-dominant axis', abs(t1["deutan"]["bias_bc_mean"]) < abs(t1["deutan"]["bias_bs_median"]), True, mode='eq')

deutan {'n': 7, 'f10': 0.264, 'n_draw': 20, 'bias_bs_median': 16.0, 'bias_bc_median': -4.0, 'bias_bs_mean': 18.571, 'bias_bc_mean': -4.714}
protan {'n': 7, 'f10': 0.136, 'n_draw': 20, 'bias_bs_median': 11.0, 'bias_bc_median': -27.0, 'bias_bs_mean': 11.857, 'bias_bc_mean': -26.429}
[OK ] 05.01 S12 ¶3 | 7 donors x 20 draws = 140 samples per candidate: produced=(7, 20)  reported=(7, 20)
[OK ] 05.02 tab:identifiability | deutan f_10: produced=0.2643  reported=0.26
[OK ] 05.03 tab:identifiability | protan f_10: produced=0.1357  reported=0.14
[OK ] 05.04 tab:identifiability | both fail the f_10 >= 0.5 criterion: produced=True  reported=True
[OK ] 05.05 tab:identifiability | deutan bias on beta_s (+16, median over donors): produced=16  reported=16
[OK ] 05.06 tab:identifiability | deutan bias on beta_c (-4.7, mean over donors; the donor median is -4.0): produced=-4.714  reported=-4.7
[OK ] 05.07 tab:identifiability | protan bias on beta_s (+11): produced=11  reported=11
[OK ] 05.08 tab:identi

### Test 2a: noise floor at a (0, 0) ground truth (tab:identifiability)
Median |beta| on each axis with the interquartile range, and the fraction recovered within 10 degrees of the origin.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 05.10 | tab:identifiability | deutan median |beta_s| (IQR) | `(22.0, 40.0)` |
| 05.11 | tab:identifiability | deutan median |beta_c| (IQR) | `(26.0, 10.5)` |
| 05.12 | tab:identifiability | deutan f_origin | `0.0` |
| 05.13 | tab:identifiability | protan median |beta_s| (IQR) | `(16.0, 17.5)` |
| 05.14 | tab:identifiability | protan median |beta_c| (IQR) | `(24.0, 9.0)` |
| 05.15 | tab:identifiability | protan f_origin | `0.0` |
| 05.16 | S12 ¶4 | effective uncertainty about 20 degrees on beta_s and 25 on beta_c (means of the two medians) | `(19, 25)` |

In [3]:
t2a = {}
for k, c in CAND.items():
    b2 = nl[c]["B2_summary"]; bs = np.abs(b2["raw_beta_s"]); bc = np.abs(b2["raw_beta_c"])
    q = lambda x: (float(np.median(x)), float(np.percentile(x, 75) - np.percentile(x, 25)))
    t2a[k] = dict(bs=q(bs), bc=q(bc), f_origin=float(np.mean((bs <= 10) & (bc <= 10))), n=b2["n"])
    print(k, t2a[k])
V.check('05.10', 'tab:identifiability | deutan median |beta_s| (IQR)', t2a["deutan"]["bs"], (22.0, 40.0), mode='pair', nd=1)
V.check('05.11', 'tab:identifiability | deutan median |beta_c| (IQR)', t2a["deutan"]["bc"], (26.0, 10.5), mode='pair', nd=1)
V.check('05.12', 'tab:identifiability | deutan f_origin', t2a["deutan"]["f_origin"], 0.0, nd=2)
V.check('05.13', 'tab:identifiability | protan median |beta_s| (IQR)', t2a["protan"]["bs"], (16.0, 17.5), mode='pair', nd=1)
V.check('05.14', 'tab:identifiability | protan median |beta_c| (IQR)', t2a["protan"]["bc"], (24.0, 9.0), mode='pair', nd=1)
V.check('05.15', 'tab:identifiability | protan f_origin', t2a["protan"]["f_origin"], 0.0, nd=2)
V.check('05.16', 'S12 ¶4 | effective uncertainty about 20 degrees on beta_s and 25 on beta_c (means of the two medians)', (round(np.mean([t2a[k]["bs"][0] for k in t2a])), round(np.mean([t2a[k]["bc"][0] for k in t2a]))), (19, 25), mode='eq')

deutan {'bs': (22.0, 40.0), 'bc': (26.0, 10.5), 'f_origin': 0.0, 'n': 140}
protan {'bs': (16.0, 17.5), 'bc': (24.0, 9.0), 'f_origin': 0.0, 'n': 140}
[OK ] 05.10 tab:identifiability | deutan median |beta_s| (IQR): produced=(22, 40)  reported=(22, 40)
[OK ] 05.11 tab:identifiability | deutan median |beta_c| (IQR): produced=(26, 10.5)  reported=(26, 10.5)
[OK ] 05.12 tab:identifiability | deutan f_origin: produced=0  reported=0
[OK ] 05.13 tab:identifiability | protan median |beta_s| (IQR): produced=(16, 17.5)  reported=(16, 17.5)
[OK ] 05.14 tab:identifiability | protan median |beta_c| (IQR): produced=(24, 9)  reported=(24, 9)
[OK ] 05.15 tab:identifiability | protan f_origin: produced=0  reported=0
[OK ] 05.16 S12 ¶4 | effective uncertainty about 20 degrees on beta_s and 25 on beta_c (means of the two medians): produced=(19, 25)  reported=(19, 25)


### Tests 2b and 2c and the BH correction (tab:identifiability)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 05.17 | tab:identifiability | deutan Test 2b rank_dist | `0.875` |
| 05.18 | tab:identifiability | protan Test 2b rank_dist | `0.875` |
| 05.19 | tab:identifiability | deutan Test 2c real loss | `-2.892` |
| 05.20 | tab:identifiability | deutan Test 2c 5% cut | `-3.136` |
| 05.21 | tab:identifiability | deutan Test 2c p | `0.167` |
| 05.22 | tab:identifiability | protan Test 2c real loss | `-1.681` |
| 05.23 | tab:identifiability | protan Test 2c 5% cut | `-3.053` |
| 05.24 | tab:identifiability | protan Test 2c p | `0.471` |
| 05.25 | S12 ¶1 | N = 1,000 permutations | `1000` |
| 05.26 | S12 ¶1 | BH over six checks: none significant | `0` |

In [4]:
for k, c in CAND.items():
    print(k, vm[c]["specificity"]["rank_distance"], vm[c]["within_subject_sig"], vm[c]["fdr_significant"])
n_bh = sum(vm[c]["fdr_significant"][t]["BH_significant"] for c in CAND.values() for t in ("identifiability", "within_subject_sig", "specificity"))
V.check('05.17', 'tab:identifiability | deutan Test 2b rank_dist', vm["S08-robust"]["specificity"]["rank_distance"], 0.875, nd=3)
V.check('05.18', 'tab:identifiability | protan Test 2b rank_dist', vm["S09-primary"]["specificity"]["rank_distance"], 0.875, nd=3)
V.check('05.19', 'tab:identifiability | deutan Test 2c real loss', vm["S08-robust"]["within_subject_sig"]["real_loss"], -2.892, nd=3)
V.check('05.20', 'tab:identifiability | deutan Test 2c 5% cut', vm["S08-robust"]["within_subject_sig"]["perm_loss_5pct"], -3.136, nd=3)
V.check('05.21', 'tab:identifiability | deutan Test 2c p', vm["S08-robust"]["within_subject_sig"]["p_perm"], 0.167, nd=3)
V.check('05.22', 'tab:identifiability | protan Test 2c real loss', vm["S09-primary"]["within_subject_sig"]["real_loss"], -1.681, nd=3)
V.check('05.23', 'tab:identifiability | protan Test 2c 5% cut', vm["S09-primary"]["within_subject_sig"]["perm_loss_5pct"], -3.053, nd=3)
V.check('05.24', 'tab:identifiability | protan Test 2c p', vm["S09-primary"]["within_subject_sig"]["p_perm"], 0.471, nd=3)
V.check('05.25', 'S12 ¶1 | N = 1,000 permutations', vm["S08-robust"]["within_subject_sig"]["n_perm"], 1000, mode='eq')
V.check('05.26', 'S12 ¶1 | BH over six checks: none significant', n_bh, 0, mode='eq')

deutan 0.875 {'PASS': False, 'p_perm': 0.16683316683316685, 'n_perm': 1000, 'real_loss': -2.8923767566919483, 'perm_loss_5pct': -3.1358200884887997, 'perm_loss_median': -2.4627266306883207} {'identifiability': {'p_value_proxy': 0.8, 'BH_significant': False}, 'within_subject_sig': {'p_value_proxy': 0.16683316683316685, 'BH_significant': False}, 'specificity': {'p_value_proxy': 0.875, 'BH_significant': False}}
protan 0.875 {'PASS': False, 'p_perm': 0.47052947052947053, 'n_perm': 1000, 'real_loss': -1.6806559032848642, 'perm_loss_5pct': -3.0525867198909524, 'perm_loss_median': -1.6372579043801374} {'identifiability': {'p_value_proxy': 0.85, 'BH_significant': False}, 'within_subject_sig': {'p_value_proxy': 0.47052947052947053, 'BH_significant': False}, 'specificity': {'p_value_proxy': 0.875, 'BH_significant': False}}
[OK ] 05.17 tab:identifiability | deutan Test 2b rank_dist: produced=0.875  reported=0.875
[OK ] 05.18 tab:identifiability | protan Test 2b rank_dist: produced=0.875  reported

### Control leave-one-out magnitude anchor (Supplementary S13)
||beta|| of each control refitted as a pseudo-CVD case under the participant's selected loss (Test 2b refits, B1), against the CVD estimate.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 05.27 | S13 ¶1 | deutan ||beta|| | `42.4` |
| 05.28 | S13 ¶1 | deutan control range low | `30.5` |
| 05.29 | S13 ¶1 | deutan control range high | `58.1` |
| 05.30 | S13 ¶1 | deutan control mean | `49.1` |
| 05.31 | S13 ¶1 | protan ||beta|| | `24.1` |
| 05.32 | S13 ¶1 | protan control range low | `23.4` |
| 05.33 | S13 ¶1 | protan control range high | `55.5` |
| 05.34 | S13 ¶1 | protan control mean | `35.7` |
| 05.35 | S13 ¶1 | both estimates fall inside the control range | `True` |

In [5]:
anchor = {}
for k, c in CAND.items():
    b1 = nl[c]["B1_summary"]; norms = np.hypot(b1["raw_beta_s"], b1["raw_beta_c"])
    anchor[k] = dict(cvd=vm[c]["specificity"]["real_distance"], lo=float(norms.min()), hi=float(norms.max()), mean=float(norms.mean()), n=len(norms))
    print(k, anchor[k])
V.check('05.27', 'S13 ¶1 | deutan ||beta||', anchor["deutan"]["cvd"], 42.4, nd=1)
V.check('05.28', 'S13 ¶1 | deutan control range low', anchor["deutan"]["lo"], 30.5, nd=1)
V.check('05.29', 'S13 ¶1 | deutan control range high', anchor["deutan"]["hi"], 58.1, nd=1)
V.check('05.30', 'S13 ¶1 | deutan control mean', anchor["deutan"]["mean"], 49.1, nd=1)
V.check('05.31', 'S13 ¶1 | protan ||beta||', anchor["protan"]["cvd"], 24.1, nd=1)
V.check('05.32', 'S13 ¶1 | protan control range low', anchor["protan"]["lo"], 23.4, nd=1)
V.check('05.33', 'S13 ¶1 | protan control range high', anchor["protan"]["hi"], 55.5, nd=1)
V.check('05.34', 'S13 ¶1 | protan control mean', anchor["protan"]["mean"], 35.7, nd=1)
V.check('05.35', 'S13 ¶1 | both estimates fall inside the control range', all(anchor[k]["lo"] <= anchor[k]["cvd"] <= anchor[k]["hi"] for k in anchor), True, mode='eq')

deutan {'cvd': 42.42640687119285, 'lo': 30.463092423455635, 'hi': 58.137767414994535, 'mean': 49.138021121075944, 'n': 7}
protan {'cvd': 24.08318915758459, 'lo': 23.40939982143925, 'hi': 55.46169849544819, 'mean': 35.66627143795189, 'n': 7}
[OK ] 05.27 S13 ¶1 | deutan ||beta||: produced=42.43  reported=42.4
[OK ] 05.28 S13 ¶1 | deutan control range low: produced=30.46  reported=30.5
[OK ] 05.29 S13 ¶1 | deutan control range high: produced=58.14  reported=58.1
[OK ] 05.30 S13 ¶1 | deutan control mean: produced=49.14  reported=49.1
[OK ] 05.31 S13 ¶1 | protan ||beta||: produced=24.08  reported=24.1
[OK ] 05.32 S13 ¶1 | protan control range low: produced=23.41  reported=23.4
[OK ] 05.33 S13 ¶1 | protan control range high: produced=55.46  reported=55.5
[OK ] 05.34 S13 ¶1 | protan control mean: produced=35.67  reported=35.7
[OK ] 05.35 S13 ¶1 | both estimates fall inside the control range: produced=True  reported=True


In [6]:
V.summary()


=== 05_identifiability: 35/35 numeric checks reproduced exactly; 0 within one unit of the last printed digit; 0 mismatch, 0 error, 0 pointer-only ===
